In [29]:
import numpy as np
import pandas as pd

пункт а

In [30]:
np.random.seed(42)
n = 50
x = np.random.uniform(-1, 1, size=(n, 5))


In [31]:
def mnk_coef(A, y):
    return np.linalg.inv(A.T @ A) @ A.T @ y


def get_r2(y, y_hat):
    rss = np.sum((y - y_hat) ** 2)
    tss = np.sum((y - np.mean(y)) ** 2)

    return 1 - rss / tss


In [32]:
for k in range(5):
    y = x[:, k]

    cols = [i for i in range(5) if i != k]

    A = x[:, cols]
    A = np.column_stack((np.ones(n), A))

    beta = mnk_coef(A, y)

    y_hat = A @ beta

    R2 = get_r2(y, y_hat)

    print(f'x{k + 1}: R^2 = {R2:.4f}')

    if R2 > 0.7:
        print('Есть мультиколлинеарность')
    else:
        print('Мультиколлинеарности нет')


x1: R^2 = 0.1172
Мультиколлинеарности нет
x2: R^2 = 0.1139
Мультиколлинеарности нет
x3: R^2 = 0.1429
Мультиколлинеарности нет
x4: R^2 = 0.0461
Мультиколлинеарности нет
x5: R^2 = 0.1000
Мультиколлинеарности нет


пункт b

In [33]:
from scipy import stats

In [34]:

eps = np.random.normal(0, 1.5, size=n)

y = 2 + 3*x[:, 0] - 2*x[:, 1] + x[:, 2] + x[:, 3] - x[:, 4] + eps

X = np.column_stack((np.ones(n), x))

p = X.shape[1]

F = X.T @ X
F_inv = np.linalg.inv(F)

beta = F_inv @ X.T @ y

y_hat = X @ beta
ost = y - y_hat

rss = np.sum(ost ** 2)

alpha = 0.05

names = ['beta0', 'beta1', 'beta2', 'beta3', 'beta4', 'beta5']

print('Уравнение регрессии:')
print(f'y = {beta[0]:.4f} + ({beta[1]:.4f})x1 + ({beta[2]:.4f})x2 + ({beta[3]:.4f})x3 + ({beta[4]:.4f})x4 + ({beta[5]:.4f})x5')
print()

for i in range(p):
    delta = abs(beta[i]) * np.sqrt(n - p) / np.sqrt(F_inv[i, i] * rss)

    p_value = 2 * (1 - stats.t.cdf(delta, df=n - p))

    print(names[i])
    print('coef =', beta[i])
    print('delta =', delta)
    print('p-value =', p_value)

    if p_value < alpha:
        print('коэффициент значим')
    else:
        print('коэффициент не значим')


Уравнение регрессии:
y = 2.0699 + (2.7552)x1 + (-2.1409)x2 + (1.0096)x3 + (0.7684)x4 + (-1.2105)x5

beta0
coef = 2.0699044106439626
delta = 8.390424661982443
p-value = 1.1171308322843743e-10
коэффициент значим
beta1
coef = 2.7551697250067524
delta = 6.320769440544633
p-value = 1.1386536558077864e-07
коэффициент значим
beta2
coef = -2.1409204420128534
delta = 4.788384224390643
p-value = 1.933162673384281e-05
коэффициент значим
beta3
coef = 1.0096160496557554
delta = 2.3143501414889167
p-value = 0.025378054152101592
коэффициент значим
beta4
coef = 0.7683526281707025
delta = 1.8711276445969203
p-value = 0.0679862752874012
коэффициент не значим
beta5
coef = -1.2104888881099156
delta = 2.779994410809928
p-value = 0.007968311477182466
коэффициент значим


пункт с

In [35]:
tss = np.sum((y - np.mean(y)) ** 2)

R2 = 1 - rss / tss

delta = (R2 * (n - p)) / ((1 - R2) * (p - 1))

p_value = 1 - stats.f.cdf(delta, dfn=p - 1, dfd=n - p)

print('R^2 =', R2)
print('delta =', delta)
print('p-value =', p_value)

if p_value < alpha:
    print('Коэффициент детерминации значим')
else:
    print('Коэффициент детерминации не значим')

R^2 = 0.6263847372343578
delta = 14.753641612120061
p-value = 1.703381657947034e-08
Коэффициент детерминации значим


пункт d

In [38]:
gamma = 0.95

psi_0 = np.array([1, 0, 0, 0, 0, 0])

y_prog = psi_0 @ beta

delta_interval = stats.t.ppf((1 + gamma) / 2, df=n - p) * np.sqrt(
    rss * (1 + psi_0 @ F_inv @ psi_0.T) / (n - p)
)

print(f'y_prog = {y_prog}')
print(f'Доверительный интервал для y_prog: [{y_prog - delta_interval}, {y_prog + delta_interval}]')

y_prog = 2.0699044106439626
Доверительный интервал для y_prog: [-1.4231622630038632, 5.562971084291789]


пункт е